## **Dataset Overview**
- **Tên dataset**: Online Retail II UCI
- **Nguồn**: UCI Machine Learning Repository
-  **Link dataset**: https://archive.ics.uci.edu/dataset/352/online+retail
- **Lĩnh vực**: E-commerce, ngành bán lẻ UK chuyên quà tặng độc đáo
- **Quy mô**: 1,067,371 giao dịch
- **Giai đoạn thời gian**: 01/12/2009 đến 09/12/2011
- **Phạm vi**: Giao dịch toàn cầu với tập trung vào thị trường UK


### Đọc dữ liệu từ file CSV


In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
from src.processors.csv_processor import CSVProcessor

RAW_PATH = '../data/raw/online_retail.csv'
EXPORT_PATH = '../data/processed/clean_transactions.csv'
TARGET_COUNTRY = 'United Kingdom'

processor = CSVProcessor(input_path=RAW_PATH)

# Đọc dữ liệu
data = processor.read()
print(f"Kích thước dữ liệu ban đầu: {data.shape}")
print(f"Số giao dịch: {data.shape[0]}")
display(data.head())

Kích thước dữ liệu ban đầu: (541909, 8)
Số giao dịch: 541909


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


###  Thông tin cơ bản về dataset
- Tên cột và kiểu dữ liệu của từng cột 
- Số lượng khách
- Top quốc gia nhiều giao dịch nhất



In [ ]:
print("Thông tin tổng quan:")
print(f"- Kích thước: {data.shape[0]:,} dòng x {data.shape[1]} cột")
print(f"- Tổng số khách: {data['CustomerID'].nunique()} khách")
print(f"- Top 3 quốc gia nhiều giao dịch nhất: \n{data['Country'].value_counts().head(3).to_string()}")
print()
data.info()


Thông tin tổng quan:
- Kích thước: 541,909 dòng x 8 cột
- Tổng số khách: 4372 khách
- Top 3 quốc gia nhiều giao dịch nhất: 
Country
United Kingdom    495478
Germany             9495
France              8557

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB


### Bảng mô tả ý nghĩa của từng cột

| Tên cột | Ý nghĩa trong dự án | Kiểu dữ liệu mong muốn | Ghi chú|
|---|---|---|---|
| InvoiceNo | Mã hóa đơn/giao dịch. Bắt đầu bằng chữ C là đơn hàng hủy | object/string |Dùng để đếm số đơn hàng và phát hiện đơn hàng bị canceled | 
| StockCode | Mã sản phẩm | object/string | Dùng để phân tích sản phẩm bán chạy |
| Description | Tên/mô tả sản phẩm | object/string | Có thể thiếu ở raw data; không phải cột cốt lõi để tính RFM |
| Quantity | Số lượng sản phẩm trong hóa đơn. Số âm là giao dịch bị hủy | interger | Phải > 0 sau khi làm sạch cho giao dịch mua hợp lệ |
| InvoiceDate | Thời điểm phát sinh hóa đơn | DateTime |Dùng để tính Recency và phân tích doanh thu theo thời gian |
| UnitPrice | Giá trên một đơn vị sản phẩm | Float  | Phải > 0 sau khi làm sạch |
| CustomerID | Mã khách hàng | string  | Bắt buộc cho Customer Segmentation/RFM. Dòng thiếu CustomerID sẽ bị loại |
| Country | Quốc gia của khách hàng/giao dịch | object/string | Dùng để EDA theo thị trường |
| TotalPrice | Doanh thu dòng giao dịch = Quantity × UnitPrice | float | Cột tạo mới sau cleaning |

### Kiểm tra xử lý dữ liệu thiếu theo cột

In [ ]:
df_uk = processor.filter_by_country(data, country=TARGET_COUNTRY)
#del data
uk_size_bf = df_uk.shape
uk_customer_bf = df_uk['CustomerID'].nunique()

subsets= ['InvoiceNo', 'StockCode']
df_uk = processor.handle_missing_values(df_uk, subsets=subsets)
missing_report= df_uk.isnull().sum().to_frame(name='Missing Values')
print("Báo cáo missing values:")
display(missing_report)
print(f"\nDữ liệu trước khi xử lý: {uk_size_bf[0]} dòng x {uk_size_bf[1]} cột")
print(f"\nDữ liệu sau khi xử lý missing: {df_uk.shape[0]} dòng x {df_uk.shape[1]} cột")

Báo cáo missing values:


,Missing Values
InvoiceNo,0
StockCode,0
Description,1454
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,133600
Country,0



Dữ liệu trước khi xử lý: 495478 dòng x 8 cột

Dữ liệu sau khi xử lý missing: 495478 dòng x 8 cột


### Kiểm tra giao dịch bị trùng 

In [ ]:
from src.utilities.convert_data_type import ConvertDataType

customer_column = ConvertDataType(column_name='CustomerID', target_type='str').convert(df_uk)
date_column = ConvertDataType(column_name='InvoiceDate', target_type='datetime').convert(df_uk)
print("Báo cáo chuyển đổi kiểu dữ liệu:")
display(df_uk[customer_column, date_column] .dtypes.to_frame('Types'))

subsets = ['InvoiceNo', 'StockCode']
df_uk = processor.handle_duplicates(df_uk, subsets=subsets)
duplicate_report = df_uk.duplicated(subset=subsets, keep=False)
sum_duplicates = duplicate_report.sum()

print(f"Số giao dịch trùng (theo InvoiceNo + StockCode): {sum_duplicates}")
display(duplicate_report.head())
print(f"\nKích thước sau khi loại bỏ trùng: {df_uk.shape}")

TypeError: ConvertDataType.convert() missing 1 required positional argument: 'df'

### Kiểm tra và xử lý giao dịch bị hủy 

In [ ]:
df_uk = processor.handle_cancelled_transactions(df_uk, key='InvoiceNo', character='C')
canceled_report = df_uk['InvoiceNo'].str.startswith('C')
print(f"Số giao dịch bị hủy: {canceled_report.sum()}")
display(canceled_report.head())
print(f"Dữ liệu sau khi loại cancelled: {df_uk.shape}")

Tổng số giao dịch bị hủy hoặc kiểm toán: 7443


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
120,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527,United Kingdom
133,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311,United Kingdom
200,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548,United Kingdom
201,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548,United Kingdom
202,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548,United Kingdom


Dữ liệu sau xử lí: 344455 dòng x 8 cột


### Kiểm tra và xử lý Quantity và UnitPrice âm

In [ ]:
# Xử lý quantity âm
df_uk, invalid_qty = processor.handle_invalid_quantity(df_uk)
print(f"Tổng số hàng có Quantity âm: {invalid_qty}")

# Xử lý price âm
df_uk, invalid_price = processor.handle_invalid_price(df_uk)
print(f"Tổng số hàng có UnitPrice âm: {invalid_price}")

print(f"\nDữ liệu sau khi loại invalid: {df_uk.shape}")

Tổng số hàng có Quantity âm: 0
Tổng số hàng có UnitPrice âm: 22


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
20731,539263,22580,ADVENT CALENDAR GINGHAM SACK,4,2010-12-16 14:36:00,0.0,16560,United Kingdom
26457,540372,22090,PAPER BUNTING RETROSPOT,24,2011-01-06 16:41:00,0.0,13081,United Kingdom
26459,540372,22553,PLASTERS IN TIN SKULLS,24,2011-01-06 16:41:00,0.0,13081,United Kingdom
31107,541109,22168,ORGANISER WOOD ANTIQUE WHITE,1,2011-01-13 15:10:00,0.0,15107,United Kingdom
47557,543599,84535B,FAIRY CAKES NOTEBOOK A6 SIZE,16,2011-02-10 13:08:00,0.0,17560,United Kingdom


Dữ liệu sau xử lí: 344433 dòng x 8 cột


### Tạo cột TotalPrice = Quantity * UnitPrice

In [ ]:
df_uk = processor.create_total_price_column(df_uk)
display(df_uk.head())

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


### Xuất cleaned dataset = clean_transactions.csv

In [ ]:
# Dataset sau khi đã được làm sạch
print(f"Kích thược UK ban đầu : {uk_size_bf[0]} dòng x {uk_size_bf[1]} cột")
print(f"Kích thước UK sau cleaning: {df_uk.shape[0]:,} dòng x {df_uk.shape[1]} cột")
print(f"Tổng số khách UK trước làm sạch: {uk_customer_bf} khách")
print(f"Tổng số khách UK sau làm sạch: {df_uk['CustomerID'].nunique()} khách")
print(f"Tỷ lệ dữ liệu bị loại bỏ: {((uk_size_bf[0] - df_uk.shape[0]) / uk_size_bf[0] * 100):.2f}%")

# Export file
output_path = "../data/processed/clean_transactions.csv"
processor.export(df_uk, output_path)

Kích thước dữ liệu trước làm sạch: 495478 dòng x 8 cột
Kích thước dữ liệu sau làm sạch: 344,433 dòng x 9 cột
Tổng số khách UK trước làm sạch: 3950 khách
Tổng số khách UK sau làm sạch: 3920 khách
Tỷ lệ dữ liệu bị loại bỏ: 30.48%


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom,15.30
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850,United Kingdom,25.50
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850,United Kingdom,11.10
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850,United Kingdom,11.10
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047,United Kingdom,54.08


Dữ liệu đã được lưu vào: ../data/processed/clean_transactions.csv
